In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import joblib
import os

PROCESSED = r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iot-combined\data\processed"

df = pd.read_csv(os.path.join(PROCESSED, "ciciomt2024_sampled.csv"))
print(f"Loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# ── Define shared features ────────────────────────────────────────────────
# CIC-IoT-2023 had 39 features, CICIoMT-2024 has 45
# Use only shared features for cross-dataset comparability

ciciot_features = joblib.load(os.path.join(PROCESSED, "ciciot2023_feature_names.pkl"))
print(f"CIC-IoT-2023 features ({len(ciciot_features)}): {ciciot_features}")

# Find shared features
all_cols = df.drop(['label', 'binary_label'], axis=1).columns.tolist()
shared = [f for f in ciciot_features if f in all_cols]
extra  = [f for f in all_cols if f not in ciciot_features]

print(f"\nShared features: {len(shared)}")
print(f"Extra in CICIoMT-2024 only: {len(extra)} — {extra}")

# Use shared features only for cross-dataset comparison
X = df[shared].values
y = df['binary_label'].values
feature_names = shared

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

In [3]:
# ── Split, scale, save ────────────────────────────────────────────────────
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Check for inf values
X = np.where(np.isinf(X), np.nan, X)
X = pd.DataFrame(X, columns=feature_names).dropna().values
y = df['binary_label'].values[:len(X)]
print(f"After inf/nan clean: {X.shape}")

# Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Class weights
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print(f"Class weights: {class_weights}")

# Save
np.save(os.path.join(PROCESSED, "ciciomt2024_X_train.npy"), X_train)
np.save(os.path.join(PROCESSED, "ciciomt2024_X_val.npy"),   X_val)
np.save(os.path.join(PROCESSED, "ciciomt2024_X_test.npy"),  X_test)
np.save(os.path.join(PROCESSED, "ciciomt2024_y_train.npy"), y_train)
np.save(os.path.join(PROCESSED, "ciciomt2024_y_val.npy"),   y_val)
np.save(os.path.join(PROCESSED, "ciciomt2024_y_test.npy"),  y_test)
joblib.dump(scaler,        os.path.join(PROCESSED, "ciciomt2024_scaler.pkl"))
joblib.dump(class_weights, os.path.join(PROCESSED, "ciciomt2024_class_weights.pkl"))
joblib.dump(feature_names, os.path.join(PROCESSED, "ciciomt2024_feature_names.pkl"))
print("\nAll files saved.")

After inf/nan clean: (500000, 38)
Train: (350000, 38), Val: (75000, 38), Test: (75000, 38)
Class weights: {np.int64(0): np.float64(18.56370001060783), np.int64(1): np.float64(0.5138399109735652)}

All files saved.
